# Agentic RFT: Tool Calling

Train `Alibaba qwen3.6-35b-a3b` version `1` with the exact successful 10/10 Tool Calling RFT contract in Sweden Central. This notebook ends after terminal monitoring and service reward metrics; it does not deploy or invoke the fine-tuned model.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.


## 1. Setup and paid-operation boundary

Copy this module's `.env.template` to a module-local `.env`, configure your own `FOUNDRY_PROJECT_ENDPOINT` and HTTPS `RFT_TOOL_SERVER_URL`, and sign in with `az login`. Your project must support agentic RFT for the selected model. Run from the cookbook directory or its notebooks directory.

Data, tool, and grader checks run locally. Executing the training cell validates your remote tool, authenticates, uploads data, and submits or resumes one paid job.


In [ ]:

import csv
import hashlib
import json
import math
import os
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from dotenv import load_dotenv

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent] if cwd.name == "notebooks" else [cwd]
ROOT = next((p for p in candidates if (p / "data" / "manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from the cookbook or its notebooks directory.")
load_dotenv(ROOT / ".env", override=False)


PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "").strip()

REGION = "Sweden Central"
MODEL = "qwen3.6-35b-a3b"
MODEL_VERSION = "1"
TRAINING_TYPE = "globalStandard"
TRAIN_PATH = ROOT / "data" / "tool-calling-train.jsonl"
VALIDATION_PATH = ROOT / "data" / "tool-calling-validation.jsonl"
EXPECTED_TRAIN_SHA256 = "11acebfc227c48fee9dd608734ac81b45544bec76c7b814c01e905783b0d6bc4"
EXPECTED_VALIDATION_SHA256 = "7835cef9a161a1a0311d6b140c782ba60e644aec54641a74746ceab1cdc23517"
TERMINAL_STATUSES = {"succeeded", "failed", "cancelled"}
OUTPUTS = ROOT / "outputs"
STATE_PATH = OUTPUTS / "submission-state.json"
TOOL_SERVER_URL = os.getenv("RFT_TOOL_SERVER_URL", "").strip()
print({"model": MODEL, "model_version": MODEL_VERSION, "training_type": TRAINING_TYPE})


## 2. Provenance and exact guards

The bundled tool-calling training and validation files preserve the original demonstration records, with checkout line endings normalized to LF for portable integrity checks and upload. Exactly 10 rows per split, fixed canonical SHA-256 hashes, the original `search_catalog` function schema, messages, and reference answers are preserved. The manifest uses source descriptions instead of private checkout paths.


In [ ]:
EXPECTED_TOOL = {
    "type": "function",
    "function": {
        "name": "search_catalog",
        "description": (
            "Search the product catalog and return matching jacket items with sku, "
            "name, and price."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Maximum number of items to return.",
                    "default": 3,
                },
            },
            "required": ["query"],
        },
    },
}

MULTI_GRADER = {
    "type": "multi",
    "name": "strict_partial_credit",
    "invalid_grade": 0.0,
    "graders": {
        "exact": {
            "type": "string_check",
            "name": "exact",
            "operation": "eq",
            "input": "{{sample.output_text}}",
            "reference": "{{item.reference_answer}}",
        },
        "fuzzy": {
            "type": "text_similarity",
            "name": "fuzzy",
            "evaluation_metric": "fuzzy_match",
            "input": "{{sample.output_text}}",
            "reference": "{{item.reference_answer}}",
        },
    },
    "calculate_output": "0.9 * exact + 0.1 * fuzzy",
}


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    return [
        json.loads(line)
        for line in path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes().replace(b"\r\n", b"\n")).hexdigest()

def validate_dataset_contract() -> tuple[list[dict[str, Any]], list[dict[str, Any]]]:
    train_rows = read_jsonl(TRAIN_PATH)
    validation_rows = read_jsonl(VALIDATION_PATH)
    assert len(train_rows) == 10
    assert len(validation_rows) == 10
    assert sha256(TRAIN_PATH) == EXPECTED_TRAIN_SHA256
    assert sha256(VALIDATION_PATH) == EXPECTED_VALIDATION_SHA256
    for split, rows in (("train", train_rows), ("validation", validation_rows)):
        for index, row in enumerate(rows):
            assert row["tools"] == [EXPECTED_TOOL], (
                f"{split} row {index} changed the search_catalog contract"
            )
            assert row["messages"]
            assert row["reference_answer"]
    return train_rows, validation_rows

train_rows, validation_rows = validate_dataset_contract()
import importlib.util

tool_spec = importlib.util.spec_from_file_location("customer_catalog_tool", ROOT / "tools" / "catalog_server.py")
tool_module = importlib.util.module_from_spec(tool_spec)
tool_spec.loader.exec_module(tool_module)
for row in train_rows + validation_rows:
    for arguments in row.get("reference_tool_args", []):
        items = tool_module.execute_search_catalog(**arguments)["items"]
        assert row["reference_answer"] in {item["sku"] for item in items}
assert MULTI_GRADER["calculate_output"] == "0.9 * exact + 0.1 * fuzzy"
assert MULTI_GRADER["invalid_grade"] == 0.0
print({"training_rows": len(train_rows), "validation_rows": len(validation_rows),
       "training_sha256": sha256(TRAIN_PATH), "validation_sha256": sha256(VALIDATION_PATH),
       "grader": MULTI_GRADER})


## 3. Visible training recipe

The service multi-grader uses 90% exact string match plus 10% fuzzy similarity, with invalid outputs scored as zero. Tool episodes allow at most 10 steps. Evaluation runs every 3 steps with 5 samples, compute multiplier 1.0, and medium reasoning effort.


In [ ]:
HYPERPARAMETERS = {
    "eval_interval": 3,
    "eval_samples": 5,
    "compute_multiplier": 1.0,
    "reasoning_effort": "medium",
}
METHOD = {
    "type": "reinforcement",
    "reinforcement": {
        "grader": MULTI_GRADER,
        "tools": [{"name": "search_catalog", "server_url": TOOL_SERVER_URL}],
        "max_episode_steps": 10,
        "hyperparameters": HYPERPARAMETERS,
    },
}
print({"model": MODEL, "version": MODEL_VERSION, "hyperparameters": HYPERPARAMETERS,
       "max_episode_steps": 10})


## 4. Validate the remote tool, upload, and submit once

Executing this cell crosses the paid-operation boundary. The request envelope and expected catalog response are checked before submission. Preserve the reusable tool implementation under `tools/`; host it at an HTTPS URL reachable by the training service. Its catalog, query behavior, and function-call envelope must match the checked-in data.

Ignored `outputs/submission-state.json` records the project, full method (including grader/tool URL), and input hashes. Rerunning resumes the recorded job. Archive state intentionally to change recipes. Job-creation POSTs have SDK retries disabled. Do not automatically retry an ambiguous job-creation failure; reconcile it in Foundry first.


In [ ]:
def validate_remote_tool(tool_server_url: str) -> None:
    request = {
        "type": "function_call",
        "id": "fc_contract_check",
        "call_id": "call_contract_check",
        "name": "search_catalog",
        "arguments": json.dumps({"query": "Urban Puffer", "top_k": 1}),
        "trace_id": "cookbook-contract-check",
        "item": {},
    }
    response = requests.post(tool_server_url, json=request, timeout=60)
    response.raise_for_status()
    payload = response.json()
    assert payload["type"] == "function_call_output"
    assert payload["id"] == request["id"]
    assert payload["call_id"] == request["call_id"]
    output = json.loads(payload["output"])
    assert output["items"][0] == {
        "sku": "JKT-URB-009",
        "name": "Urban Puffer",
        "price": 149.5,
    }

def wait_for_file(client: Any, file_id: str, poll_seconds: float) -> Any:
    deadline = time.monotonic() + 600
    while True:
        uploaded = client.files.retrieve(file_id)
        status = str(uploaded.status).casefold()
        print(f"file {file_id}: {status}")
        if status in {"processed", "completed"}:
            return uploaded
        if status in {"error", "failed", "expired", "cancelled"}:
            raise RuntimeError(f"File {file_id} ended as {status}: {uploaded}")
        if time.monotonic() >= deadline:
            raise TimeoutError(f"File {file_id} exceeded 600 seconds")
        time.sleep(poll_seconds)

def upload(client: Any, path: Path, poll_seconds: float) -> Any:
    with path.open("rb") as handle:
        uploaded = client.files.create(
            file=(path.name, path.read_bytes().replace(b"\r\n", b"\n"), "application/jsonl"),
            purpose="fine-tune",
        )
    return wait_for_file(client, uploaded.id, poll_seconds)

job = None
from urllib.parse import urlparse
import requests
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

if not PROJECT_ENDPOINT:
    raise RuntimeError("Set FOUNDRY_PROJECT_ENDPOINT in your .env or process environment.")
if urlparse(TOOL_SERVER_URL).scheme != "https" or not urlparse(TOOL_SERVER_URL).netloc:
    raise ValueError("RFT_TOOL_SERVER_URL must be an HTTPS URL reachable by the training service.")
validate_remote_tool(TOOL_SERVER_URL)
credential = DefaultAzureCredential(process_timeout=120)
project = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
client = project.get_openai_client()
OUTPUTS.mkdir(parents=True, exist_ok=True)
equivalence = {
    "project_endpoint": PROJECT_ENDPOINT, "model": MODEL, "model_version": MODEL_VERSION,
    "training_type": TRAINING_TYPE, "method": METHOD,
    "training_sha256": sha256(TRAIN_PATH), "validation_sha256": sha256(VALIDATION_PATH),
}
state = json.loads(STATE_PATH.read_text(encoding="utf-8")) if STATE_PATH.exists() else None
if state and state.get("equivalence") != equivalence:
    raise RuntimeError("Saved state belongs to another recipe or project; archive it before a new run.")
if state and state.get("job_id"):
    job = client.fine_tuning.jobs.retrieve(state["job_id"])
    print(f"Resuming job {job.id}: {job.status}")
else:
    train_file = upload(client, TRAIN_PATH, 5)
    validation_file = upload(client, VALIDATION_PATH, 5)
    job = client.with_options(max_retries=0).fine_tuning.jobs.create(
        model=MODEL, training_file=train_file.id, validation_file=validation_file.id,
        suffix="tool-calling-rft", method=METHOD,
        extra_body={"trainingType": TRAINING_TYPE},
    )
    state = {"equivalence": equivalence, "job_id": job.id,
             "training_file_id": train_file.id, "validation_file_id": validation_file.id}
    STATE_PATH.write_text(json.dumps(state, indent=2) + "\n", encoding="utf-8")
    print(f"Submitted job {job.id}: {job.status}")


## 5. Monitor with a timeout and download service reward metrics

A failed or cancelled job raises an error. A timeout leaves the recorded job available to resume; it does not cancel training.


In [ ]:
def event_dict(event: Any) -> dict[str, Any]:
    if hasattr(event, "to_dict"):
        return event.to_dict()
    return json.loads(event.to_json())

def wait_for_job(
    client: Any,
    job_id: str,
    events_path: Path,
    poll_seconds: float,
) -> Any:
    seen: set[str] = set()
    records: list[dict[str, Any]] = []
    deadline = time.monotonic() + 21_600
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"{datetime.now(timezone.utc).isoformat()} {job_id}: {job.status}")
        for event in reversed(list(client.fine_tuning.jobs.list_events(job_id, limit=100))):
            record = event_dict(event)
            key = json.dumps(record, sort_keys=True, default=str)
            if key not in seen:
                seen.add(key)
                records.append(record)
                print(f"  {record.get('level', 'info')}: {record.get('message', '')}")
        events_path.write_text(
            json.dumps(records, indent=2, default=str),
            encoding="utf-8",
        )
        if str(job.status).casefold() in TERMINAL_STATUSES:
            if str(job.status).casefold() != "succeeded":
                raise RuntimeError(f"Training ended as {job.status}: {job.error}")
            return job
        if time.monotonic() >= deadline:
            raise TimeoutError(f"Job {job_id} exceeded 21600 seconds; rerun to resume.")
        time.sleep(poll_seconds)

def download_result_files(client: Any, job: Any, run_dir: Path) -> list[Path]:
    paths: list[Path] = []
    for file_id in job.result_files or []:
        response = client.files.content(file_id)
        content = getattr(response, "content", None)
        if content is None and hasattr(response, "read"):
            content = response.read()
        if not isinstance(content, bytes):
            raise RuntimeError(f"Result file {file_id} did not return bytes")
        path = run_dir / f"{file_id}.csv"
        path.write_bytes(content)
        paths.append(path)
    return paths

def finite_float(value: Any) -> float | None:
    if value in (None, ""):
        return None
    parsed = float(value)
    if not math.isfinite(parsed):
        raise ValueError(f"Non-finite service metric: {value!r}")
    return parsed

def summarize_curve(rows: list[dict[str, str]], column: str) -> dict[str, Any]:
    points = [
        {"step": int(row["step"]), "value": finite_float(row.get(column))}
        for row in rows
        if finite_float(row.get(column)) is not None
    ]
    if not points:
        return {"metric": column, "points": 0, "trend": "unavailable", "values": []}
    start = points[0]["value"]
    end = points[-1]["value"]
    delta = end - start
    best = max(points, key=lambda point: point["value"])
    return {
        "metric": column,
        "points": len(points),
        "start": start,
        "end": end,
        "delta": delta,
        "best": best["value"],
        "best_step": best["step"],
        "trend": "improving" if delta > 0 else "declining" if delta < 0 else "flat",
        "values": points,
    }

run_dir = OUTPUTS / job.id
run_dir.mkdir(parents=True, exist_ok=True)
job = wait_for_job(client, job.id, run_dir / "events.json", 60)
result_paths = download_result_files(client, job, run_dir)
metric_rows = []
for path in result_paths:
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        metric_rows.extend(csv.DictReader(handle))
train_curve = summarize_curve(metric_rows, "train_mean_reward")
validation_curve = summarize_curve(metric_rows, "full_valid_mean_reward")
if not train_curve["values"] and not validation_curve["values"]:
    raise RuntimeError("No training or validation reward metrics returned by the service.")
summary = {"status": job.status, "model": MODEL, "model_version": MODEL_VERSION,
           "training_reward": train_curve, "validation_reward": validation_curve}
(run_dir / "run-summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
client.close()
project.close()
credential.close()


## 6. Training-only conclusion

Use only the current job's service `train_mean_reward` and `full_valid_mean_reward` curves. The prior reward of `1.0` is reference context, not evidence for this run. The notebook does not deploy or invoke the trained model.

> Actual results may vary by model version, data, configuration, region availability, and service conditions.
